# Cerebellar

In [2]:
#@title: cerebellar
class ClimbingFiber(nn.Module):
class CerebellarCNN(nn.Module):
    def __init__(self, n_grc=8, n_pkg=32, n_motor=4, tau=cfg.tau, T=32, std = 0.0):
        super(CerebellarCNNAC2, self).__init__()
        self.critic = nn.Sequential(
            nn.Conv2d(in_channels = 1, out_channels = n_grc, kernel_size= 5, stride = 1, padding = 'valid', padding_mode = 'zeros', bias = False),
            neuron.LIFNode(tau=tau, surrogate_function=surrogate.ATan(), detach_reset=True),
            nn.Flatten(),
            nn.Linear(in_features = n_grc*(6**2), out_features = 1, bias = False),
            #neuron.LIFNode(tau=tau, surrogate_function=surrogate.ATan(), detach_reset=True)
            NonSpikingLIFNode(tau = tau)
        )
        self.actor = nn.Sequential(
            nn.Conv2d(in_channels = 1, out_channels = n_grc, kernel_size= 5, stride = 1, padding = 'valid', padding_mode = 'zeros', bias = False),
            neuron.LIFNode(tau=tau, surrogate_function=surrogate.ATan(), detach_reset=True),
            nn.Flatten(),
            nn.Linear(in_features = n_grc*6**2, out_features = n_motor, bias = False),
            neuron.LIFNode(tau=tau, surrogate_function=surrogate.ATan(), detach_reset=True)
            #NonSpikingLIFNode(tau = tau)
        )
        self.log_std = nn.Parameter(torch.ones(1, 4) * std)
        self.T = T
        for m in self.modules():
            if isinstance(m, nn.Linear):
                torch.nn.init.uniform_(m.weight.data)
            
            if isinstance(m, nn.Conv2d):
                torch.nn.init.uniform_(m.weight.data)
        
    def forward(self, x):
        # ... (SNN 순전파 코드)
        #for t in range(self.T):
        critic = self.critic(x)
        actor = self.actor(x)
        value = self.critic[-1].v
        mu = self.actor[-1].v
        std = self.log_std.exp().expand_as(mu)
        dist = Normal(mu, std)
        #if DEBUG: print(f"[TRAIN] actor:{actor}, critic:{critic}, actor potential:{self.actor[-1].v}, critic potential:{self.critic[-1].v},")
        return dist, value

class CerbellarNet(nn.Module):
    def __init__(self, cfg):
        super(CerbellarNet, self).__init__()
        self.mf2grc = nn.Conv2d(in_channels = 1, out_channels = cfg.n_grc, kernel_size= 5, stride = 1, padding = 'valid', padding_mode = 'zeros', bias = False)
        self.grc2pkj = nn.Linear(in_features = cfg.n_grc*(6**2), out_features = cfg.n_pkj, bias = False)
        self.pkj2motor = nn.Linear(in_features = cfg.n_pkj, out_features = cfg.n_motor, bias = False)
        self.lif_grc = neuron.LIFNode(tau=cfg.tau, surrogate_function=surrogate.ATan(), detach_reset=True)
        self.lif_pkj = neuron.LIFNode(tau=cfg.tau, surrogate_function=surrogate.ATan(), detach_reset=True)
        self.lif_motor = neuron.LIFNode(tau=cfg.tau, surrogate_function=surrogate.ATan(), detach_reset=True)
        self.T = 16  # SNN 시뮬레이션 시간 (스텝 수)
    
    def forward(self, x, motor): # added inhibition from (t-1) motor
        for t in range(self.T):
            x = self.mf2grc(x)
            x = self.lif_grc(x)
            x = x.view(x.size(0), -1)  # Flatten
            x = self.grc2pkj(x)
            x = self.lif_pkj(x)
            x = self.pkj2motor(x)
            x = self.lif_motor(x - motor)  # Inhibition from previous motor output
        return x
      '''  

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 64)